# NN_02 — normalization

`BUILD_SPEC.md` §7, row 02: the per-feature normalization policy from §4/§5, reusing `apply_cross_sectional_rank_normalization`, with before/after distributions. This notebook **verifies the policy before any model is built**. It performs **no model training** and produces **no performance result**.

**The policy** (`nn_common.RANK_NORMALIZED`, `nn_common.normalize_cycle`): exactly five **raw-level** features — `market_cap`, `volume`, `net_debt`, `adjusted_net_debt`, `analyst_target_price` — are replaced by their percentile rank within their own cycle's cross-section, scaled to `[0, 1]`, by the strategy's production function `cross_sectional.apply_cross_sectional_rank_normalization`, the only rank transform used anywhere in this series. Every other stock-level feature — growth, margin, ratio, momentum, beta, volatility, trend — is **passed through untouched**, as are the macro and categorical features. Nothing else is done to the observations: no standardization, no imputation, no clipping.

**The decision this records.** `BUILD_SPEC.md` §4's code block ("rank everything except `NEVER_RANK`") and §5 rule 2 ("growth/ratio features are already differenced — pass them through, standardize at most") conflicted. The conflict was resolved on 2026-09-08 by honoring rule 2: rank-normalization is limited to raw-level features, where cross-quarter dollar/share scale drift is the actual problem, so the network does not lose the cross-quarter level and sign of growth/ratio features before it gets a chance to learn from them. The measurements that motivated the decision are reproduced below.

**Expected consequence, verified rather than assumed:** these five are the same features the production feature pipeline already ranks (`config.cross_sectional_rank_features`), so on the cached slice `normalize_cycle` should change **zero** entries. That is the headline check.

**Data mode: REAL DATA.** Everything below is computed from the cached real-data slice (`rd.slice_bundle()`, fingerprint verified in `NN_00`). No synthetic, simulated, or approximated value appears anywhere; if the raw exports are absent, every computing cell prints the explicit unavailability notice and does nothing.

Cycle **2023-10-01**, the same cycle as `NN_01`. Kernel: `atlasquant-venv` (= `.venv/bin/python`).

In [1]:
import nn_common as nc

rd = nc.rd
AVAILABLE = nc.provenance_banner("NN_02_normalization.ipynb")

neural_ranker / NN_02_normalization.ipynb
DATA MODE: REAL DATA.

Source: Bloomberg CSV exports under this repository's gitignored raw-data
root for multi_factor_ranking_ml -- 1,520 tickers, quarterly fundamentals
from 1980 onward, plus daily closes and macro series.

Every number below is computed from that real data through production
AtlasQuant code. No synthetic, simulated, imputed, or approximated values
are used anywhere in this notebook series.

Bounded slice: quarterly evaluation cycles 2020-10-01 .. 2024-01-01
(a genuine subset of the history, chosen so each notebook runs quickly --
not a downsample or an approximation of a longer run).

SOURCE MANIFEST (SHA-256 of every raw CSV the pipeline reads)
  raw root: /Users/gavinhussey/Downloads/AtlasQuant/data/raw/multi_factor_ranking_ml
  fundamentals_quarterly.csv      281.3 MB  2026-08-31 23:01:10  3315e7104e640fc790b6e12c66b0f6a122cf66bb97a8ed3124abaced0c86b48a
  Close_Price.csv                  67.6 MB  2026-08-31 23:07:42  2723

libraries         : python=3.14.5, torch=2.13.0, numpy=2.5.2, pandas=3.0.5, scipy=1.18.0, lightgbm=4.7.0


## The per-feature policy

Five disjoint groups covering all 71 features. `normalize_cycle` is a one-line delegation to the production function — printed here so there is no doubt about what the "policy" executes. The policy must be *exactly* the pipeline's own five names, in the pipeline's own order.

In [2]:
import inspect

from atlas_quant.strategies.multi_factor_ranking_ml.feature_domain import FEATURE_NAMES

PIPELINE_RANKED = tuple(rd.config().cross_sectional_rank_features)

print("RANK_NORMALIZED ", len(nc.RANK_NORMALIZED), nc.RANK_NORMALIZED)
print("PIPELINE_RANKED ", len(PIPELINE_RANKED), PIPELINE_RANKED)
print("PASS_THROUGH    ", len(nc.PASS_THROUGH), "stock-level features passed through byte-identically")
print("  of which NEVER_RANK", len(nc.NEVER_RANK), nc.NEVER_RANK)
print("MACRO           ", len(nc.MACRO), "-> never passed to the transform; carried once per cycle as m (6,)")
print("CATEGORICAL     ", len(nc.CATEGORICAL), "-> never passed to the transform; embeddings")
print()
print(inspect.getsource(nc.normalize_cycle))

assert nc.RANK_NORMALIZED == PIPELINE_RANKED, "policy must be exactly the five pipeline-ranked raw-level features"
assert nc.RANK_NORMALIZED == ("market_cap", "volume", "net_debt", "adjusted_net_debt", "analyst_target_price")
assert set(nc.RANK_NORMALIZED) | set(nc.PASS_THROUGH) == set(nc.STOCK_CONTINUOUS)
assert not set(nc.RANK_NORMALIZED) & set(nc.PASS_THROUGH)
assert set(nc.NEVER_RANK) <= set(nc.PASS_THROUGH)
assert not set(nc.RANK_NORMALIZED) & (set(nc.MACRO) | set(nc.CATEGORICAL) | set(nc.NEVER_RANK))
assert len(nc.RANK_NORMALIZED) + len(nc.PASS_THROUGH) + len(nc.MACRO) + len(nc.CATEGORICAL) == len(FEATURE_NAMES) == 71
assert "apply_cross_sectional_rank_normalization" in inspect.getsource(nc.normalize_cycle)
assert len(nc.PASS_THROUGH) == 58

RANK_NORMALIZED  5 ('market_cap', 'volume', 'net_debt', 'adjusted_net_debt', 'analyst_target_price')
PIPELINE_RANKED  5 ('market_cap', 'volume', 'net_debt', 'adjusted_net_debt', 'analyst_target_price')
PASS_THROUGH     58 stock-level features passed through byte-identically
  of which NEVER_RANK 8 ('volatility_20d', 'volatility_30d', 'volatility_63d', 'volatility_90d', 'vol_20d', 'vol_63d', 'vol_ratio', 'beta')
MACRO            6 -> never passed to the transform; carried once per cycle as m (6,)
CATEGORICAL      2 -> never passed to the transform; embeddings

def normalize_cycle(observations: Sequence[FeatureObservation]) -> tuple[FeatureObservation, ...]:
    """Apply the per-feature policy to exactly ONE cycle's cross-section.

    Delegates the rank transform entirely to the strategy's own
    ``apply_cross_sectional_rank_normalization`` (the only sanctioned
    implementation -- it enforces the single-cycle scope, preserves NaN, and
    records an audit record on every observation)

### The pass-through features, by family

For reporting only (the policy itself is the five-name tuple above): every one of the 58 pass-through features belongs to one of these families, all of which §5 rule 2 (growth/margin/ratio/momentum/trend, already differenced or scale-free) or rule 1 (volatility/beta) says must not be ranked.

In [3]:
FAMILIES = nc.FEATURE_FAMILIES  # reporting-only grouping, defined once in nn_common
covered = [f for members in FAMILIES.values() for f in members]
for family, members in FAMILIES.items():
    print(f"{family:<32}{len(members):>3}  {', '.join(members)}")
print(f"{'total':<32}{len(covered):>3}")
assert sorted(covered) == sorted(nc.PASS_THROUGH) and len(covered) == len(set(covered)) == 58

volatility / beta (NEVER_RANK)    8  volatility_20d, volatility_30d, volatility_63d, volatility_90d, vol_20d, vol_63d, vol_ratio, beta
growth rates                     20  revenue_yoy_growth, revenue_qoq_growth, operating_income_yoy_growth, operating_income_qoq_growth, net_income_yoy_growth, net_income_qoq_growth, diluted_eps_yoy_growth, diluted_eps_qoq_growth, operating_cash_flow_yoy_growth, operating_cash_flow_qoq_growth, total_assets_yoy_growth, total_assets_qoq_growth, total_debt_yoy_growth, total_debt_qoq_growth, stockholders_equity_yoy_growth, stockholders_equity_qoq_growth, diluted_share_count_yoy_growth, diluted_share_count_qoq_growth, shares_outstanding_yoy_growth, shares_outstanding_qoq_growth
growth acceleration               4  revenue_growth_acceleration, operating_income_growth_acceleration, eps_growth_acceleration, operating_cash_flow_growth_acceleration
margins                           3  operating_margin, net_margin, operating_cash_flow_margin
margin changes (bps)    

## Before and after, cycle 2023-10-01

`before` is the cross-section exactly as the cached slice holds it; `after` is `normalize_cycle(before)`. Both are turned into `CycleTensors` by `NN_01`'s builder, whose own guards (mask = `missing_features`, one macro value per cycle, categoricals in vocabulary) therefore run on the normalized observations too.

The production function appends an audit record to every observation it touches. The `before` observations already carry one such record from the feature pipeline (5 features, cross-section size N); `after` carries a second, and it must name the same 5 features and the same N. That record is evidence, from the function itself, of exactly what was ranked and over how many names.

In [4]:
if rd.require_data():
    import numpy as np

    bundle = rd.slice_bundle()
    CYCLE = bundle.cycles[-2]
    assert CYCLE.quarter_start.isoformat() == "2023-10-01", "expected the same cycle as NN_01"
    before_obs = bundle.feature_results[CYCLE.quarter_start].observations
    labeled = bundle.labeled_quarters[CYCLE.quarter_start]

    after_obs = nc.normalize_cycle(before_obs)

    B = nc.build_cycle_tensors(CYCLE, before_obs, labeled)
    A = nc.build_cycle_tensors(CYCLE, after_obs, labeled)
    N = B.n
    print("cycle          ", CYCLE.quarter_start, "cutoff", CYCLE.cutoff)
    print("N              ", N)
    print("X before/after ", B.X.shape, A.X.shape)
    assert A.n == B.n == len(before_obs) == len(after_obs)
    assert A.instrument_ids == B.instrument_ids, "the transform must not reorder the cross-section"

    stages_before = [r.stage for r in before_obs[0].audit_trail.records]
    stages_after = [r.stage for r in after_obs[0].audit_trail.records]
    print("audit stages, before:", stages_before)
    print("audit stages, after: ", stages_after)
    upstream = [r for r in before_obs[0].audit_trail.records if r.stage == "cross_sectional_rank_normalization"][0]
    ours = after_obs[0].audit_trail.records[-1]
    print("upstream pipeline record :", upstream.message, "->", upstream.data["normalized_features"])
    print("this notebook's record   :", ours.message, "->", ours.data["normalized_features"])
    sizes = set()
    for obs in after_obs:
        record = obs.audit_trail.records[-1]
        assert record.stage == "cross_sectional_rank_normalization"
        assert tuple(record.data["normalized_features"]) == nc.RANK_NORMALIZED == tuple(upstream.data["normalized_features"])
        sizes.add(record.data["cross_section_size"])
    print("cross_section_size on every after-observation:", sizes, "| upstream:", upstream.data["cross_section_size"])
    assert sizes == {N} == {upstream.data["cross_section_size"]}

cycle           2023-10-01 cutoff 2023-09-30
N               1493
X before/after  (1493, 63) (1493, 63)
audit stages, before: ['point_in_time_selection', 'cross_sectional_rank_normalization']
audit stages, after:  ['point_in_time_selection', 'cross_sectional_rank_normalization', 'cross_sectional_rank_normalization']
upstream pipeline record : percentile-rank normalized 5 feature(s) within this cycle's 1493-name cross-section -> ['market_cap', 'volume', 'net_debt', 'adjusted_net_debt', 'analyst_target_price']
this notebook's record   : percentile-rank normalized 5 feature(s) within this cycle's 1493-name cross-section -> ['market_cap', 'volume', 'net_debt', 'adjusted_net_debt', 'analyst_target_price']
cross_section_size on every after-observation: {1493} | upstream: 1493


## Acceptance: every `NEVER_RANK` feature is byte-identical

Compared as raw bytes of the float64 column (so `NaN` positions count too), before vs after, and again value-by-value at the observation level.

In [5]:
if rd.require_data():
    print(f"{'feature':<18}{'byte-identical':>15}{'NaN':>6}{'min':>10}{'median':>10}{'max':>10}")
    for name in nc.NEVER_RANK:
        j = nc.STOCK_CONTINUOUS.index(name)
        same = B.X[:, j].tobytes() == A.X[:, j].tobytes()
        col = A.X[:, j]
        print(f"{name:<18}{str(same):>15}{int(np.isnan(col).sum()):>6}{np.nanmin(col):>10.3f}{np.nanmedian(col):>10.3f}{np.nanmax(col):>10.3f}")
        assert same, f"{name} was altered by the transform"
    for b_obs, a_obs in zip(before_obs, after_obs):
        for name in nc.NEVER_RANK:
            vb, va = b_obs.features[name], a_obs.features[name]
            assert (vb != vb and va != va) or vb == va
    print("\nall 8 NEVER_RANK features byte-identical before vs after: True")

feature            byte-identical   NaN       min    median       max
volatility_20d               True    36     7.562    27.023   232.964
volatility_30d               True    38     9.551    29.478   207.338
volatility_63d               True    40    10.117    30.629   173.186
volatility_90d               True    41    10.778    31.915   296.121
vol_20d                      True    42     0.082     0.272     2.443
vol_63d                      True    49     0.099     0.302     1.821
vol_ratio                    True    49     0.299     0.918     1.513
beta                         True    44   -21.391     0.896    24.533

all 8 NEVER_RANK features byte-identical before vs after: True


## Acceptance: every growth / margin / ratio / momentum / trend feature is byte-identical

All 58 pass-through features, family by family. These keep their raw scale and sign; scaling them for the network is `NN_04`'s input-layer decision and must use fixed, non-fitted maps (a per-cycle statistic would discard the same cross-quarter level ranking discards).

In [6]:
if rd.require_data():
    print(f"{'family':<32}{'features':>9}{'byte-identical':>15}{'NaN':>7}{'min':>13}{'median':>11}{'max':>13}")
    for family, members in FAMILIES.items():
        idx = [nc.STOCK_CONTINUOUS.index(name) for name in members]
        same = all(B.X[:, j].tobytes() == A.X[:, j].tobytes() for j in idx)
        block = A.X[:, idx]
        print(
            f"{family:<32}{len(members):>9}{str(same):>15}{int(np.isnan(block).sum()):>7}"
            f"{np.nanmin(block):>13.4g}{np.nanmedian(block):>11.4g}{np.nanmax(block):>13.4g}"
        )
        assert same, f"a {family} feature was altered by the transform"
    pass_idx = [nc.STOCK_CONTINUOUS.index(name) for name in nc.PASS_THROUGH]
    assert B.X[:, pass_idx].tobytes() == A.X[:, pass_idx].tobytes()
    for b_obs, a_obs in zip(before_obs, after_obs):
        for name in nc.PASS_THROUGH:
            vb, va = b_obs.features[name], a_obs.features[name]
            assert (vb != vb and va != va) or vb == va
    print("\nall 58 pass-through features byte-identical before vs after, at tensor and observation level: True")

family                           features byte-identical    NaN          min     median          max
volatility / beta (NEVER_RANK)          8           True    339       -21.39      11.54        296.1
growth rates                           20           True    319        -4130   0.006955         6417
growth acceleration                     4           True     82        -1342  -0.009326        382.1
margins                                 3           True     19       -148.4     0.1263        7.256
margin changes (bps)                    4           True     42   -1.481e+06      20.51    7.619e+05
valuation ratios                        3           True    324       0.0242      4.151         1580
other ratios                            7           True     81       -961.9     0.2174         2783
legacy trends                           6           True     76        -1920     0.0101        765.1
price momentum                          3           True    134      -0.7989    0.06787    

## Acceptance: macro untouched, still one `(6,)` vector

Each macro feature must still take exactly one distinct *finite* value across the cross-section, equal to its value before, and `m` must still be a single `(6,)` row — `CycleTensors` has no per-row macro field at all, so tiling is impossible by construction, and that is asserted rather than assumed.

In [7]:
if rd.require_data():
    print(f"{'macro feature':<20}{'distinct (after)':>17}{'finite':>8}{'value':>12}{'== before':>11}")
    for j, name in enumerate(nc.MACRO):
        column = np.array([obs.features[name] for obs in after_obs], dtype=np.float64)
        distinct = np.unique(column)
        finite = bool(np.isfinite(column).all())
        print(f"{name:<20}{len(distinct):>17}{str(finite):>8}{distinct[0]:>12.4f}{str(distinct[0] == B.m[j]):>11}")
        assert len(distinct) == 1 and finite and distinct[0] == B.m[j] == A.m[j]
    print()
    print("m shape after:", A.m.shape, "| ndim:", A.m.ndim, "| equal to before:", np.array_equal(A.m, B.m))
    assert A.m.shape == (6,) and A.m.ndim == 1 and np.array_equal(A.m, B.m)
    assert not any(field.startswith("macro") and field != "macro_names" for field in nc.CycleTensors.__dataclass_fields__), "no per-row macro tensor exists"

macro feature        distinct (after)  finite       value  == before
fed_funds_rate                      1    True      5.3300       True
hy_credit_oas                       1    True      3.9400       True
ust_10y_yield                       1    True      4.5711       True
ust_2y_yield                        1    True      5.0436       True
vix                                 1    True     17.5200       True
yield_curve_10y_2y                  1    True     -0.4725       True

m shape after: (6,) | ndim: 1 | equal to before: True


## Acceptance: missingness preserved, categoricals untouched, labels and row order untouched

`NaN` in, `NaN` out: the transform must not create or remove a single missing value, and the builder's mask must be identical. `sector_enc` and `quarter_num` are never handed to the transform and must be byte-identical at the observation level; `sector`, `quarter`, `y` and the row order must be identical at the tensor level.

In [8]:
if rd.require_data():
    nan_same = np.array_equal(np.isnan(B.X), np.isnan(A.X))
    mask_same = np.array_equal(B.mask, A.mask)
    print("NaN locations identical        ", nan_same)
    print("mask identical                 ", mask_same)
    print("missing count before / after   ", int(B.mask.sum()), "/", int(A.mask.sum()))
    print("rows with any missing, before / after", int(B.mask.any(axis=1).sum()), "/", int(A.mask.any(axis=1).sum()))
    assert nan_same and mask_same and int(B.mask.sum()) == int(A.mask.sum()) == 1542

    cat_same = all(
        b_obs.features[name] == a_obs.features[name]
        for b_obs, a_obs in zip(before_obs, after_obs)
        for name in nc.CATEGORICAL
    )
    print("sector_enc / quarter_num identical on every observation:", cat_same)
    print("sector tensor identical        ", np.array_equal(B.sector, A.sector))
    print("quarter tensor identical       ", np.array_equal(B.quarter, A.quarter))
    print("y identical                    ", np.array_equal(B.y, A.y))
    print("row order identical            ", A.instrument_ids == B.instrument_ids)
    print("missing_features tuples identical", all(b.missing_features == a.missing_features for b, a in zip(before_obs, after_obs)))
    assert cat_same
    assert np.array_equal(B.sector, A.sector) and np.array_equal(B.quarter, A.quarter) and np.array_equal(B.y, A.y)
    assert A.instrument_ids == B.instrument_ids
    assert all(b.missing_features == a.missing_features for b, a in zip(before_obs, after_obs))

NaN locations identical         True
mask identical                  True
missing count before / after    1542 / 1542
rows with any missing, before / after 315 / 315
sector_enc / quarter_num identical on every observation: True
sector tensor identical         True
quarter tensor identical        True
y identical                     True
row order identical             True
missing_features tuples identical True


## Headline: re-ranking the five already-ranked features changes zero entries

The feature pipeline already replaced these five with `(rank − 1)/(n − 1)` over this same cross-section. Ranking such a column again should reproduce it exactly — same order, same tie groups, same `n` — but that is a claim about floating-point arithmetic on real data, so it is measured. If it holds, `normalize_cycle` is a no-op on the whole of `X` for this slice.

In [9]:
if rd.require_data():
    print(f"{'feature':<22}{'byte-identical':>15}{'changed entries':>17}{'max |diff|':>12}{'non-missing':>13}{'min':>6}{'max':>6}")
    total_changed = 0
    for name in nc.RANK_NORMALIZED:
        j = nc.STOCK_CONTINUOUS.index(name)
        b_col, a_col = B.X[:, j], A.X[:, j]
        same = b_col.tobytes() == a_col.tobytes()
        ok = ~np.isnan(b_col)
        changed = int((b_col[ok] != a_col[ok]).sum())
        max_diff = float(np.max(np.abs(b_col[ok] - a_col[ok]))) if ok.any() else 0.0
        total_changed += changed
        print(f"{name:<22}{str(same):>15}{changed:>17}{max_diff:>12.3g}{int(ok.sum()):>13}{np.nanmin(a_col):>6.1f}{np.nanmax(a_col):>6.1f}")
        assert same and changed == 0
        assert np.nanmin(a_col) == 0.0 and np.nanmax(a_col) == 1.0
    whole_x_same = B.X.tobytes() == A.X.tobytes()
    print()
    print("changed entries across the five ranked features:", total_changed)
    print("changed entries across all 63 columns of X     :", int(np.sum((B.X != A.X) & ~np.isnan(B.X))))
    print("X byte-identical before vs after (normalize_cycle is a no-op on this slice):", whole_x_same)
    assert total_changed == 0 and whole_x_same

feature                byte-identical  changed entries  max |diff|  non-missing   min   max
market_cap                       True                0           0         1464   0.0   1.0
volume                           True                0           0         1460   0.0   1.0
net_debt                         True                0           0         1488   0.0   1.0
adjusted_net_debt                True                0           0         1474   0.0   1.0
analyst_target_price             True                0           0         1453   0.0   1.0

changed entries across the five ranked features: 0
changed entries across all 63 columns of X     : 0
X byte-identical before vs after (normalize_cycle is a no-op on this slice): True


## Distributions

**The five ranked features**, before (already ranked upstream) and after (identical): decile-bucket counts of the rank are flat by construction, ties excepted — `net_debt` and `adjusted_net_debt` carry tie groups (many names at exactly zero net debt) that share an average rank.

**The pass-through features** keep their raw scale. Their per-family range in this cycle is shown for `NN_04`'s scaling decision; `NN_01` has the full 63-column table.

In [10]:
if rd.require_data():
    def text_hist(values, edges, width=40):
        counts, _ = np.histogram(values, bins=edges)
        peak = max(counts.max(), 1)
        for lo, hi, count in zip(edges[:-1], edges[1:], counts):
            print(f"  [{lo:>5.2f}, {hi:>5.2f})  {int(count):>5}  {'#' * int(round(width * count / peak))}")

    for name in nc.RANK_NORMALIZED:
        j = nc.STOCK_CONTINUOUS.index(name)
        a_col = A.X[:, j]; a_col = a_col[~np.isnan(a_col)]
        print(f"{name}  (n = {len(a_col)}, distinct = {len(np.unique(a_col))}, min {a_col.min():.3f}, mean {a_col.mean():.4f}, max {a_col.max():.3f}; before == after)")
        text_hist(a_col, np.linspace(0.0, 1.0, 11))
        print()

market_cap  (n = 1464, distinct = 1459, min 0.000, mean 0.5000, max 1.000; before == after)
  [ 0.00,  0.10)    147  ########################################
  [ 0.10,  0.20)    146  ########################################
  [ 0.20,  0.30)    146  ########################################
  [ 0.30,  0.40)    147  ########################################
  [ 0.40,  0.50)    146  ########################################
  [ 0.50,  0.60)    146  ########################################
  [ 0.60,  0.70)    147  ########################################
  [ 0.70,  0.80)    146  ########################################
  [ 0.80,  0.90)    146  ########################################
  [ 0.90,  1.00)    147  ########################################

volume  (n = 1460, distinct = 1460, min 0.000, mean 0.5000, max 1.000; before == after)
  [ 0.00,  0.10)    146  ########################################
  [ 0.10,  0.20)    146  ########################################
  [ 0.20,  0.30)    146  ##

## What the pass-through policy preserves (§5 rules 1 and 2, measured over 14 cycles)

A within-cycle rank keeps the *order* of names and throws away the *level*. The two tables below are the evidence behind the policy decision recorded at the top:

- **Rule 1 (volatility/beta):** the per-cycle median of the volatility features moves substantially across the slice. That level is what not-ranking preserves — the reason the 2026-09-08 study found ranking volatility erased the gain.
- **Rule 2 (growth/margin/ratio):** the per-cycle median and the share of *negative* values of the growth features also move across the slice. Had they been ranked, a name at rank 0.5 in a quarter where the median growth is negative would be indistinguishable from a name at rank 0.5 in a quarter where the median is strongly positive. Under the policy adopted here they are passed through, so the network sees the sign and the level.

In [11]:
if rd.require_data():
    ALL_BEFORE = nc.bundle_cycle_tensors(bundle)

    def per_cycle(name, stat):
        j = nc.STOCK_CONTINUOUS.index(name)
        return [stat(ALL_BEFORE[c.quarter_start].X[:, j]) for c in bundle.cycles]

    print("Rule 1 -- volatility / beta, per-cycle raw median (preserved: not ranked):")
    rule1 = ("volatility_90d", "vol_63d", "beta", "vol_ratio")
    print(f"{'quarter_start':<14}" + "".join(f"{n:>16}" for n in rule1))
    for i, c in enumerate(bundle.cycles):
        print(f"{c.quarter_start.isoformat():<14}" + "".join(f"{per_cycle(n, np.nanmedian)[i]:>16.4f}" for n in rule1))
    for n in rule1:
        meds = per_cycle(n, np.nanmedian)
        print(f"  {n:<16} median ranges {min(meds):.4f} .. {max(meds):.4f}  (x{max(meds) / min(meds):.2f})")

    print()
    print("Rule 2 -- growth features, per-cycle raw median and share of NEGATIVE values (preserved: not ranked):")
    rule2 = ("revenue_yoy_growth", "net_income_yoy_growth", "diluted_eps_yoy_growth", "operating_margin_yoy_change_bps")
    print(f"{'quarter_start':<14}" + "".join(f"{n[:22]:>24}" for n in rule2))
    print(f"{'':<14}" + "".join(f"{'median':>14}{'neg %':>10}" for _ in rule2))
    for i, c in enumerate(bundle.cycles):
        row = f"{c.quarter_start.isoformat():<14}"
        for n in rule2:
            j = nc.STOCK_CONTINUOUS.index(n)
            col = ALL_BEFORE[c.quarter_start].X[:, j]
            col = col[~np.isnan(col)]
            row += f"{np.median(col):>14.4f}{100 * (col < 0).mean():>9.1f}%"
        print(row)
    for n in rule2:
        j = nc.STOCK_CONTINUOUS.index(n)
        negs = []
        for c in bundle.cycles:
            col = ALL_BEFORE[c.quarter_start].X[:, j]
            col = col[~np.isnan(col)]
            negs.append(100 * (col < 0).mean())
        meds = per_cycle(n, np.nanmedian)
        print(f"  {n:<34} median {min(meds):+.4f} .. {max(meds):+.4f};  negative share {min(negs):.1f}% .. {max(negs):.1f}%")

Rule 1 -- volatility / beta, per-cycle raw median (preserved: not ranked):
quarter_start   volatility_90d         vol_63d            beta       vol_ratio
2020-10-01             84.2880          0.6038          1.1620          0.8367
2021-01-01             42.4415          0.3611          1.1876          0.9909
2021-04-01             38.5370          0.3777          1.2446          0.7267
2021-07-01             35.8020          0.3635          1.2548          0.9420
2021-10-01             31.5915          0.2708          1.2466          0.9103
2022-01-01             27.8430          0.2786          1.2400          0.9500
2022-04-01             30.5390          0.3073          1.2167          0.9157
2022-07-01             36.4880          0.3642          1.0904          0.9891
2022-10-01             41.2060          0.3973          1.0446          1.0087
2023-01-01             36.6740          0.3368          1.0752          1.0091
2023-04-01             37.9170          0.3734          

## All 14 cycles, one cross-section per call

The transform is applied through `bundle_cycle_tensors`'s per-cycle hook. The production function is wrapped with a call spy for this pass, so the number of calls and the size of each call's input are recorded facts rather than an assumption: 14 calls, each receiving exactly one cycle's observations (all sharing that cycle's cutoff), each with the five-name policy. Every invariant above is then re-checked on every cycle, including the no-op on the whole of `X`.

In [12]:
if rd.require_data():
    from unittest.mock import patch

    with patch.object(
        nc, "apply_cross_sectional_rank_normalization", wraps=nc.apply_cross_sectional_rank_normalization
    ) as spy:
        ALL_AFTER = nc.bundle_cycle_tensors(bundle, transform=nc.normalize_cycle)

    print("calls to apply_cross_sectional_rank_normalization:", spy.call_count)
    print(f"{'quarter_start':<14}{'N':>6}{'call input':>12}{'one cutoff':>12}{'features':>10}{'X same':>8}{'mask same':>11}{'macro 1':>9}{'cat same':>10}{'y same':>8}{'order same':>12}")
    for call, cycle in zip(spy.call_args_list, bundle.cycles):
        call_obs, call_names = call.args
        b, a = ALL_BEFORE[cycle.quarter_start], ALL_AFTER[cycle.quarter_start]
        one_cutoff = len({o.data_cutoff for o in call_obs}) == 1 and call_obs[0].data_cutoff.date() == cycle.cutoff
        x_same = b.X.tobytes() == a.X.tobytes()
        mask_same = np.array_equal(b.mask, a.mask)
        macro_one = a.m.shape == (6,) and np.array_equal(a.m, b.m) and np.isfinite(a.m).all()
        cat_same = np.array_equal(b.sector, a.sector) and np.array_equal(b.quarter, a.quarter)
        y_same = (a.y is None and b.y is None) or (a.y is not None and b.y is not None and np.array_equal(a.y, b.y))
        order_same = a.instrument_ids == b.instrument_ids
        print(
            f"{cycle.quarter_start.isoformat():<14}{b.n:>6}{len(call_obs):>12}{str(one_cutoff):>12}{len(call_names):>10}"
            f"{str(x_same):>8}{str(mask_same):>11}{str(macro_one):>9}{str(cat_same):>10}{str(y_same):>8}{str(order_same):>12}"
        )
        assert len(call_obs) == b.n and one_cutoff and tuple(call_names) == nc.RANK_NORMALIZED
        assert x_same and mask_same and macro_one and cat_same and y_same and order_same
    assert spy.call_count == len(bundle.cycles) == 14
    print("\nnormalize_cycle changed zero entries in every cycle of the slice: True")

calls to apply_cross_sectional_rank_normalization: 14
quarter_start      N  call input  one cutoff  features  X same  mask same  macro 1  cat same  y same  order same
2020-10-01      1423        1423        True         5    True       True     True      True    True        True
2021-01-01      1430        1430        True         5    True       True     True      True    True        True
2021-04-01      1436        1436        True         5    True       True     True      True    True        True
2021-07-01      1451        1451        True         5    True       True     True      True    True        True
2021-10-01      1462        1462        True         5    True       True     True      True    True        True
2022-01-01      1468        1468        True         5    True       True     True      True    True        True
2022-04-01      1469        1469        True         5    True       True     True      True    True        True
2022-07-01      1472        1472        Tr

## Point-in-time check

Stated explicitly, per `CLAUDE.md` and `BUILD_SPEC.md` §6. **The transform is applied cycle by cycle, using no other quarter.** Every normalization statistic here is a rank within one cycle's own cross-section; no quarter is pooled with any other, and no scaling parameter is fit anywhere. What was checked, not assumed:

1. The only rank transform executed is `apply_cross_sectional_rank_normalization`, which by its own contract reads nothing but the observations handed to it. The spy above recorded 14 calls, one per cycle, each receiving exactly that cycle's N observations, all carrying that cycle's cutoff.
2. Normalizing the 2023-10-01 cross-section in isolation (the `after_obs` at the top of this notebook) and normalizing it inside the 14-cycle pass produce byte-identical tensors.
3. The production function's audit record on every observation names the cross-section size it ranked over — N, never more — and matches the upstream pipeline's own record for the same cycle.

`NN_03` performs the direct audit `BUILD_SPEC.md` §7 row 03 asks for: building cycle *t* with and without the cycles after *t* present yields identical tensors.

In [13]:
if rd.require_data():
    alone = A
    in_pass = ALL_AFTER[CYCLE.quarter_start]
    identical = all(getattr(alone, k).tobytes() == getattr(in_pass, k).tobytes() for k in ("X", "mask", "sector", "quarter", "m", "y"))
    print("2023-10-01 normalized alone vs inside the 14-cycle pass, byte-identical:", identical)
    assert identical
    sizes = {obs.audit_trail.records[-1].data["cross_section_size"] for obs in after_obs}
    print("cross-section size recorded by the production function on every observation:", sizes, "== N:", sizes == {N})
    assert sizes == {N}
    print("point-in-time checks 1-3: passed")

2023-10-01 normalized alone vs inside the 14-cycle pass, byte-identical: True
cross-section size recorded by the production function on every observation: {1493} == N: True
point-in-time checks 1-3: passed


## What this established

Every number below was printed by a cell above in this executed run. **This notebook trained no model and produced no performance result**; it verified a normalization policy.

**The policy is exactly the five raw-level features**, `market_cap`, `volume`, `net_debt`, `adjusted_net_debt`, `analyst_target_price` — asserted equal, name for name and in order, to the production pipeline's `cross_sectional_rank_features`. The other 58 stock-level features are passed through: 8 volatility/beta, 20 growth rates, 4 growth accelerations, 3 margins, 4 margin changes in bps, 3 valuation ratios, 7 other ratios, 6 legacy trends, 3 price-momentum features. `normalize_cycle` is a one-line delegation to `apply_cross_sectional_rank_normalization`, the only rank transform that runs.

**It is called once per cycle with exactly that cycle's cross-section.** The call spy recorded 14 calls over the 14-cycle pass, each receiving N observations (1,423 .. 1,494) all carrying that cycle's cutoff, each with the five-name policy. On 2023-10-01 the production function's own audit record on every observation says 5 features ranked within a 1,493-name cross-section — identical to the upstream pipeline's record for the same cycle.

**Headline: `normalize_cycle` is a measured no-op on this slice.** Re-ranking the five already-ranked features changed **0 entries** (max |diff| 0, byte-identical) on cycle 2023-10-01; across all 63 columns of `X`, 0 entries changed and `X` is byte-identical before vs after; the same held in every one of the 14 cycles. The five columns sit exactly in `[0, 1]` with mean 0.5000 (1,464 / 1,460 / 1,488 / 1,474 / 1,453 non-missing).

**Acceptance checks, all on cycle 2023-10-01 (N = 1,493) and repeated on all 14 cycles:**
- All 8 `NEVER_RANK` features byte-identical before vs after (float64 bytes, `NaN` positions included), at tensor and observation level.
- All 58 growth / margin / ratio / momentum / trend / volatility features byte-identical, family by family.
- Each of the 6 macro features still has exactly 1 distinct, finite value, equal to before; `m` is one `(6,)` row (ndim 1); `CycleTensors` has no per-row macro field.
- `sector_enc` and `quarter_num` identical on every observation; `sector`, `quarter`, `y` tensors identical; row order identical.
- `NaN` locations identical, mask identical, 1,542 missing before and after, 315 rows with any missing before and after, `missing_features` tuples unchanged.

**What the pass-through policy preserves, measured over 14 cycles.** Volatility/beta (rule 1): the per-cycle median of `volatility_90d` runs 27.8 .. 84.3 (×3.03), `vol_63d` 0.27 .. 0.60 (×2.25). Growth (rule 2): `revenue_yoy_growth` median −5.4% .. +23.1% with the share of negative-growth names 13.0% .. 62.4%; `net_income_yoy_growth` median −18.0% .. +70.0%, negative share 21.9% .. 60.0%; `operating_margin_yoy_change_bps` median −184 .. +435, negative share 26.5% .. 61.5%. A within-cycle rank would map each of those medians to 0.5 regardless of sign; under this policy the network sees the level and the sign.

**Point-in-time, checked.** The transform is applied cycle by cycle using no other quarter; normalizing 2023-10-01 alone and inside the 14-cycle pass gives byte-identical tensors; no scaling parameter is fit anywhere. `NN_03` performs the direct with-and-without-later-cycles audit.

**Consequence for `NN_04`.** Because 58 of the 63 columns of `X` keep their raw scale (up to ~1.5 × 10⁶ in the bps features, thousands in growth rates), the network's input layer must scale them with **fixed, non-fitted** elementwise maps — never a per-cycle statistic, which would discard the same cross-quarter level this policy just preserved, and never a globally fitted one (`BUILD_SPEC.md` §6). That is `NN_04`'s decision and it will be verified there.